<a href="https://colab.research.google.com/github/Klynnae31/Healthcare-Claims-Analytics-Dashboard/blob/main/Healthcare_Claims_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openpyxl

In [ ]:
import pandas as pd
import random
from datetime import datetime, timedelta

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.listdir('/content/drive')

['.shortcut-targets-by-id', 'MyDrive', '.Trash-0', '.Encrypted']

In [ ]:
claims = pd.read_excel('/content/drive/MyDrive/Healthcare Data Analytics Project.xlsx')
claims.head()

,Claim_ID,Member ID,Provider Name,Provider Type,State,Service Date,Diagnosis,Procedure,Claim amount,Paid Amount,...,Denial Reason,Length of stay,Inpatient/Outpatient,Facility,Region,Quarter,Year,Month,Claim Category,Risk Score
0,CLM100001,MEM500001,John Hopkins Hospital,Hospital,MD,2026-01-01,Hypertension,Office Visit,325.50,287.25,...,NaN,0,OP,JHH,NE,Q1,2026,Jan,Professional,2.4
1,CLM100002,MEM500002,Medstar Health,Hospital,MD,2026-01-02,Type 2 Diabetes,Hemoglobin A1C,185.25,165.00,...,NaN,0,OP,Medstar,NE,Q2,2026,Feb,Professional,3.1
2,CLM100003,MEM500003,Mercy Medical Center,Hospital,PA,2026-01-03,Chest Pain,ECG,642.10,0.00,...,Missing prior auth,0,OP,Mercy,NE,Q3,2026,Mar,Professional,2.8
3,CLM100004,MEM500004,Lifebridge Health,Hospital,VA,2026-01-04,Asthma,Chest X-ray,410.00,365.75,...,NaN,0,OP,Lifebridge,S,Q4,2026,Apr,Professional,1.9
4,CLM100005,MEM500005,University of Maryland MC,Hospital,MD,2026-01-05,Low Back Pain,MRI Lumbar Spine,1850.00,1620.00,...,NaN,0,OP,UMMC,NE,Q1,2026,May,Professional,3.7


In [ ]:
import random
import pandas as pd
from datetime import timedelta

new_claims = []

provider_names = [
    "Johns Hopkins Hospital",
    "MedStar Health",
    "University of Maryland Medical Center",
    "Kaiser Permanente",
    "Mercy Medical Center"
]

provider_types = [
    "Hospital",
    "Primary Care",
    "Specialist",
    "Urgent Care"
]

states = [
    "MD",
    "VA",
    "DC",
    "PA",
    "DE"
]

claim_statuses = [
    "Paid",
    "Denied",
    "Pending"
]

diagnoses = [
    "Hypertension",
    "Diabetes",
    "Asthma",
    "COVID-19",
    "Back Pain"
]

procedures = [
    "Office Visit",
    "MRI",
    "CT Scan",
    "Blood Test",
    "Physical Therapy"
]

for i in range(10000):

    row = claims.sample(1).iloc[0].copy()

    row["Claim_ID"] = f"C{100001+i}"
    row["Member_ID"] = f"M{random.randint(10000,99999)}"

    row["Provider_Name"] = random.choice(provider_names)
    row["Provider_Type"] = random.choice(provider_types)

    row["State"] = random.choice(states)

    row["Diagnosis"] = random.choice(diagnoses)
    row["Procedure"] = random.choice(procedures)

    row["Claim_Amount"] = round(random.uniform(100,15000),2)

    row["Paid_Amount"] = round(
        row["Claim_Amount"] * random.uniform(.65,.98),2
    )

    row["Claim_Status"] = random.choice(claim_statuses)

    new_claims.append(row)

generated_claims = pd.DataFrame(new_claims)

generated_claims.head()

,Claim_ID,Member ID,Provider Name,Provider Type,State,Service Date,Diagnosis,Procedure,Claim amount,Paid Amount,...,Year,Month,Claim Category,Risk Score,Member_ID,Provider_Name,Provider_Type,Claim_Amount,Paid_Amount,Claim_Status
2,C100001,MEM500003,Mercy Medical Center,Hospital,MD,2026-01-03,Hypertension,Office Visit,642.10,0.00,...,2026,Mar,Professional,2.8,M48469,University of Maryland Medical Center,Specialist,2473.83,2241.24,Denied
2,C100002,MEM500003,Mercy Medical Center,Hospital,DC,2026-01-03,COVID-19,Blood Test,642.10,0.00,...,2026,Mar,Professional,2.8,M64875,Mercy Medical Center,Specialist,1704.91,1387.38,Pending
3,C100003,MEM500004,Lifebridge Health,Hospital,DE,2026-01-04,Back Pain,Office Visit,410.00,365.75,...,2026,Apr,Professional,1.9,M36054,Johns Hopkins Hospital,Hospital,905.60,762.66,Denied
3,C100004,MEM500004,Lifebridge Health,Hospital,PA,2026-01-04,Back Pain,Physical Therapy,410.00,365.75,...,2026,Apr,Professional,1.9,M97244,Johns Hopkins Hospital,Hospital,1356.81,1023.23,Denied
1,C100005,MEM500002,Medstar Health,Hospital,MD,2026-01-02,Diabetes,Physical Therapy,185.25,165.00,...,2026,Feb,Professional,3.1,M73095,Mercy Medical Center,Primary Care,9594.34,7726.22,Pending


In [ ]:
generated_claims.shape

(10000, 42)

In [ ]:
generated_claims.to_excel(
    "/content/drive/MyDrive/Healthcare_Claims_10K.xlsx",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!


In [ ]:
import sqlite3

conn = sqlite3.connect("healthcare_claims.db")

generated_claims.to_sql(
    "claims",
    conn,
    if_exists="replace",
    index=False
)

print("Database created successfully!")

Database created successfully!


In [ ]:
query = """
SELECT COUNT(*) AS Total_Claims
FROM claims;
"""
pd.read_sql(query, conn)

,Total_Claims
0,10000


In [ ]:
query = """
SELECT
    Provider_Name,
    COUNT(*) AS Total_Claims
FROM claims
GROUP BY Provider_Name
ORDER BY Total_Claims DESC;
"""

pd.read_sql(query, conn)

,Provider_Name,Total_Claims
0,Mercy Medical Center,2031
1,University of Maryland Medical Center,2015
2,Kaiser Permanente,1990
3,Johns Hopkins Hospital,1986
4,MedStar Health,1978


In [ ]:
query = """
SELECT
    Provider_Name,
    ROUND(SUM(Claim_Amount),2) AS Total_Claim_Amount
FROM claims
GROUP BY Provider_Name
ORDER BY Total_Claim_Amount DESC;
"""

pd.read_sql(query, conn)

,Provider_Name,Total_Claim_Amount
0,Mercy Medical Center,15614928.18
1,Johns Hopkins Hospital,15258333.91
2,Kaiser Permanente,15152041.51
3,University of Maryland Medical Center,15037737.47
4,MedStar Health,15022889.85
